In [ ]:
# Install dependencies
!pip install -q transformers datasets scikit-learn accelerate

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Upload dataset_v2.jsonl from local machine
from google.colab import files
uploaded = files.upload()  # Upload dataset_v2.jsonl
import shutil
for name in uploaded:
    shutil.move(name, f'/content/{name}')
    print(f'Uploaded: {name}')

In [ ]:
import json
import random
from collections import Counter

random.seed(42)

# Load dataset
data = []
with open('/content/dataset_v2.jsonl') as f:
    for line in f:
        entry = json.loads(line)
        text = entry.get('text', '').strip()
        label = entry.get('label', -1)
        if len(text) < 50 or label not in (0, 1, 2, 3):
            continue
        # Binary: human(0,3) → 0, ai(1,2) → 1
        binary_label = 0 if label in (0, 3) else 1
        data.append({'text': text, 'label': binary_label})

print(f'Total: {len(data)} entries')
label_dist = Counter(d['label'] for d in data)
print(f'Human: {label_dist[0]}, AI: {label_dist[1]}')

from sklearn.model_selection import train_test_split
train_data, eval_data = train_test_split(data, test_size=0.1, random_state=42, stratify=[d['label'] for d in data])
print(f'Train: {len(train_data)}, Eval: {len(eval_data)}')

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

BASE_MODEL = 'microsoft/deberta-v3-base'
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize(examples):
    return tokenizer(examples['text'], truncation=True, max_length=512)

train_ds = Dataset.from_list(train_data).map(tokenize, batched=True, remove_columns=['text'])
eval_ds = Dataset.from_list(eval_data).map(tokenize, batched=True, remove_columns=['text'])
print(f'Tokenized: train={len(train_ds)}, eval={len(eval_ds)}')

In [ ]:
import numpy as np
import torch
from sklearn.metrics import accuracy_score
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2)
model.float()  # DeBERTa fp16 explodes

# Auto batch size + bf16 detection
mem = torch.cuda.get_device_properties(0).total_memory
if mem > 70e9: batch = 64
elif mem > 30e9: batch = 32
else: batch = 16

use_bf16 = torch.cuda.is_bf16_supported()
print(f'GPU: {torch.cuda.get_device_name(0)}, Memory: {mem/1e9:.0f}GB, Batch: {batch}, bf16: {use_bf16}')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds)}

OUTPUT_DIR = '/content/detector_v3'

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=4,
    per_device_train_batch_size=batch,
    per_device_eval_batch_size=batch * 2,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    logging_steps=50,
    fp16=False,
    bf16=use_bf16,  # A100=True, T4=False (auto-detect)
    warmup_ratio=0.1,
    report_to='none',
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
from sklearn.metrics import classification_report

results = trainer.evaluate()
print(f'Final eval accuracy: {results["eval_accuracy"]:.4f}')

preds = trainer.predict(eval_ds)
pred_labels = np.argmax(preds.predictions, axis=-1)
true_labels = preds.label_ids
print(classification_report(true_labels, pred_labels, target_names=['human', 'ai']))

In [ ]:
import os
SAVE_DIR = '/content/drive/MyDrive/ai-detector/detector_v3'
os.makedirs(SAVE_DIR, exist_ok=True)

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f'Model saved to {SAVE_DIR}')
print('Download from Google Drive or use the cell below to download directly.')

In [ ]:
# Zip and download
import shutil
shutil.make_archive('/content/detector_v3', 'zip', '/content/detector_v3')
files.download('/content/detector_v3.zip')